# Cell Type Atlas Validation

<img 
    src="./assets/01_Combined_Figure.png" 
    alt="Atlas Overview Figure"
    align="center" 
    style="border: 2px solid #ccc; border-radius: 8px; padding: 5px; width: 100%; box-shadow: 0px 4px 8px rgba(0,0,0,0.1);">


## Introduction
This workflow covers cell-type atlas validation of multiple single-cell RNA samples:

1. Visualize UMAP features of cell clustering to determine if cell types are well-separated.
2. Visualize cell type distribution within each sample to detect sample-associated differences.
3. Visualize marker genes for each cell type to validate cell type assignments.

### What is a cell atlas good for?
Typically, a single-cell atlas should convey the categories of cell types found in a dataset, how these categories are distributed in individual tissue or patient samples, and the key defining marker genes for each of the cell type categories. The goal is generally to act as a reference that allows future researchers to easily compare how their own samples associate with the atlas cell-types and expressed genes profile.

### About the data in this workflow
In this tutorial we will look at data from [Gleeson 2023](https://www.nature.com/articles/s41590-023-01504-2), "Conserved transcriptional connectivity of regulatory T cells in the tumor microenvironment informs new combination cancer therapy strategies", which is [available from cellxgene](https://cellxgene.cziscience.com/collections/efd94500-1fdc-4e28-9e9f-a309d0154e21). 

Though this workflow will focus on a benign prostate dataset, the anndata (`.h5ad`) source file should be easily swappable with any number of cell-type atlases from CZI cellxgene or other sources with the following components:

* pre-processed AnnData atlas, with `adata.obsm` for:
  * UMAP or other dimensionality reduction embeddings (e.g. `adata.obsm['X_umap']`, see [scanpy docs](https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html) on how to run clustering)
  * cell typings for each cell (e.g. `adata.obs['cell_type']` or equivalent)
  * sample name for each cell (e.g. `adata.obs['sample_name']` or equivalent)
* Mapping of cell type --> list of marker genes


# Imports and Configuration

In [ ]:
import anndata as ad
import holoviews as hv
import panel as pn
import hvplot.pandas  # noqa
import scanpy as sc
import pooch

from hv_anndata import Dotmap, ManifoldMap

hv.extension("bokeh")
sc.settings.set_figure_params(dpi=100, facecolor="white")
pn.extension('jsoneditor')

## Loading and Inspecting the Data

<div class="admonition alert alert-warning">
    <p class="admonition-title" style="font-weight:bold">Warning</p>
    If the data was not previously downloaded, the following will download ~600 MB the first time it is run.
</div>

In [ ]:
anndata_file_path = pooch.retrieve(
    url="https://datasets.cellxgene.cziscience.com/ad4aac9c-28e6-4a1f-ab48-c4ae7154c0cb.h5ad",
    fname="ad4aac9c-28e6-4a1f-ab48-c4ae7154c0cb.h5ad",
    known_hash="00ee1a7d9dbb77dc5b8e27d868d3c371f1f53e6ef79a18e5f1fede166b31e2eb",
    path="data-download"
)
anndata_file_path

In [ ]:
adata = ad.read_h5ad(anndata_file_path)
adata

## Explore Clusters Annotated by Cell-Type and Sample

Let's start by visualizing the UMAP clustering with cell type coloring to explore the type-assignment patterns.

**Questions to keep in mind:**
- How are different cell types distributed in our spatial embedding, and do they form distinct clusters?
- Are there any unexpected mixing patterns between cell types?

Let's start by taking a look at a static matplotlib-based UMAP visualization defined in the scanpy package:

In [ ]:
sc.pl.umap(adata, color="cell_type")

That's a useful start! But there are clearly some cluster density patterns that we are missing with so many overlapping points. It would be great to also interactively zoom in, see cluster labels, and hover over parts of the cluster to drill into the details. To acheive this, let's now use HoloViz running with the Bokeh backend to return an interactive `ManifoldMap`. This application includes controls for applying Datashader rasterization, which will dynamically aggregate points per pixel as we change the zoom/range to give us more information about the patterns of density.

<div class="admonition alert alert-info">
    <p class="admonition-title" style="font-weight:bold">Note</p>
    You can pass `show_widgets=False` to hide the side panel of widgets in the app below.
</div>

In [ ]:
ManifoldMap(adata=adata, reduction='X_umap', color_by='cell_type', width=400, height=400)

What can we see so far? For one, it looks like some of the assigned cell types form multiple clusters (e.g. 'epithelial cell') . It's possible that these are cell subtypes, but it's also possible that there are differences in each sample that contribute to the separate of the same cell type into distinct clusters. To resolve this, let's `color_by` each sample and compare.

For now, focus on the `epithelial cell` type clusters from the above plot (upper right quadrant) and then compare the same clusters in the plot below. What do we see?

In [ ]:
ManifoldMap(adata=adata, reduction='X_umap', color_by='donor_id', width=400, height=400)

<div class="admonition alert alert-info">
    <p class="admonition-title" style="font-weight:bold">Note</p>
    To view other features to color by, clear the text in the 'Color by' widget input field, or switch to the 'Variables' tab to color by expression of a particular gene.
</div>

From this comparison, we can see that some of the `epithelial cell` clusters exclusively contain cells from a single sample (e.g. donor 'HTA8_1029'), indicating that there might be a donor/sample-specific influence ('batch effect') on the clustering for this particular cell type.

However, for the rest of the data, we have found well-defined cell type clusters with minimal batch effects, as indicated by distinct color regions in cell-type-colored view, but a distributed mix of sample colors in donor/sample-colored view.

<div class="admonition alert alert-info">
    <p class="admonition-title" style="font-weight:bold">Note</p>
    This particular dataset has additional dimensionality `Reduction` methods applied; use the widget to take a look (e.g. `X_tsne`).
</div>

## Explore Cell-Type Distribution Per Sample

Next, we'll visualize cell type distribution within each sample to look for sample-associated differences. While allowing for biological variation, we want to check for highly inconsistent cell type proportions in the samples.

**Questions to keep in mind:**
- What is the relative abundance of each cell type across samples?
- Are there any notable sample-specific variations?

Let's start by counting the data entries per `donor_id` and `cell_type` combination.

In [ ]:
cell_type_counts = adata.obs.groupby(['donor_id', 'cell_type'], observed=False).size().reset_index(name='count')
cell_type_counts

We can now feed this resulting dataframe into a HoloViz hvPlot `Bars` element and set some options for useability:

In [ ]:
cell_type_counts.hvplot.bar(
    x='donor_id',
    y='count',
    by='cell_type',
    stacked=True,
    height=500,
    responsive=True,
    rot=45,
    cmap="Category10",
    title='Cell Count by Donor and Cell Type',
).opts(active_tools=[])

We could also easily plot these distributions normalized by the total count per donor:

In [ ]:
cell_type_counts['donor_total'] = cell_type_counts.groupby('donor_id', observed=False)['count'].transform('sum')
cell_type_counts['proportion'] = cell_type_counts['count'] / cell_type_counts['donor_total'] * 100
cell_type_counts

In [ ]:
cell_type_counts.hvplot.bar(
    x='donor_id',
    y='proportion',
    by='cell_type',
    stacked=True,
    height=500,
    responsive=True,
    rot=45,
    cmap="Category10",
    title='Cell Count by Donor and Cell Type',
).opts(active_tools=[])

From the above plots, we can see that we have several cell types for each of the included samples. Although a visual check is sufficient for our exploratory workflow purposes, a potential extension may be to quantify the consistency of cell type proportions across samples.

## Explore Gene Profile Per Cell Type

Finally, we want to look into the different marker genes associated with each cell type cluster to validate cell type assignments.

**Questions to keep in mind:**
- Do canonical markers adequately define our cell types?
- Are there any marker genes showing unexpected expression patterns?

Let's start by defining some marker genes. 

In [ ]:
marker_genes = {
    "ActivatedVEC": ["Bcl3", "Noct", "Relb", "Tnf", "Cerl2", "Cc40", "Irf5", "Csf1", "NiKb2", "Icosl", "Egr2", "Dll1", "Pim1", "Irf1", "Icam1", "Fgf2", "Tank", "I16", "Tgif1", "Ninj1", "Tnip1"],
    "Angiogenesis": ["LPI", "Cd36", "Miga2", "Tap1", "Wars1", "Cd74", "Lyбe", "Gbp6", "Ido1", "Ciita", "Oas2", "Vegfa", "Thod", "Slco2a1", "Jup", "Icam2", "Lima1", "Cldn5", "Pardog", "Cd47", "Fmol I", "Alas1", "Bmpr2", "Sptbnt", "Smad6", "Sema3c"],
    "Hypoxia": ["Klf6", "Nfil", "Bhlhe40", "Maff", "Serpine1", "Plaur", "Tnfaip3", "Icam1", "Nfkbia Junb", "Hbegf", "Rel", "Relb", "Fosl2", "Hmox1", "Timp3", "Irf8", "Batf3", "Nikbiz", "Pvr", "Ccr7", "Stat3"],
    "EndMT": ["Emp3", "Serpina3", "Psmg4", "Cd63", "Il1r1", "Lgmn", "Csrp2", "Len2", "Cfb", "Lgals4", "Npm3", "Traf4", "Kpnb1", "Timp1", "Gda", "Ch25", "Tgm2", "Prkca", "Csrp2", "Ngf", "Ammecr1"]
}

We'll need to convert the gene HUGO symbols to ensemble labels given the current dataset's use of ensemble labels.

In [ ]:
import requests

MYGENE_QUERY_URI = "https://mygene.info/v3/query?fields=ensembl.gene&dotfield=true&size=1&from=0&fetch_all=false&facet_size=10&entrezonly=false&ensemblonly=false"
MAX_GENES_PER_GROUP = 8

adata_gene_set = set(adata.var_names)
marker_ensemble = {}
for signature, symbols in marker_genes.items():
    query = {
      "q": symbols,
      "scopes": "symbol",
      "species": [
        "human"
      ],
      "fields": "ensemble.gene"
    }
    response = requests.post(MYGENE_QUERY_URI, json=query)

    ens_genes = []
    for gene in response.json():
        eg = gene.get("ensembl.gene", [])
        if isinstance(eg, str):
            eg = [eg]
        ens_genes += list(set(eg) & adata_gene_set)
    
    marker_ensemble[signature] = ens_genes[:MAX_GENES_PER_GROUP]

In [ ]:
marker_ensemble

Let's start by producing a static `dotplot` with scanpy to see at a glance how various gene groupings align with cell type.

In [ ]:
sc.pl.dotplot(adata, marker_ensemble, groupby='cell_type', expression_cutoff=0.0)

We can also produce a similar plot in HoloViz, with the added functionality of being able to edit the included genes through widgets. 

In [ ]:
Dotmap(adata=adata, marker_genes=marker_ensemble, groupby="cell_type", max_dot_size=20, expression_cutoff=0.0)

### Expected Output/Evaluation:

We expect high expression of markers in their assigned cell types with minimal expression elsewhere.